# Corpus Data Preparation

Cleans and normalises the raw corpus, builds sentence embeddings, **automatically** finds
the best number of topic clusters, and writes `corpus_nodes_clustered.toml`.

**Workflow:**
1. Load all JSON files
2. Filter bad scrapes (`MIN_BODY_LEN`)
3. Clean / normalise text (encoding artefacts, boilerplate)
4. Embed with `sentence-transformers` — **full body, no truncation**
5. UMAP 2-D visualisation coloured by original prefix
6. Auto-select best k via silhouette score over `K_RANGE`
7. Apply KMeans; show UMAP coloured by cluster
8. Inspect clusters; fill `CLUSTER_TO_NODE` in config cell
9. Export `corpus_nodes_clustered.toml`


## 0 — Configuration

All tunable parameters live here. Re-run the notebook after changing any value.


In [ ]:
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
CORPUS_DIR      = Path("corpus")
EMBED_CACHE     = Path(".embeddings_cache.npy")  # delete to force re-encoding
OUTPUT_TOML     = Path("corpus_nodes_clustered.toml")
OUTPUT_MANIFEST = Path("cluster_assignments.json")

# ── Quality filter ─────────────────────────────────────────────────────────
MIN_BODY_LEN = 200  # characters after cleaning; shorter = likely bad scrape

# ── Embedding ──────────────────────────────────────────────────────────────
# Full body is encoded — no length cap.
# Swap model for quality vs. speed trade-off:
#   fast : all-MiniLM-L6-v2
#   best : all-mpnet-base-v2
EMBED_MODEL = "all-mpnet-base-v2"

# ── UMAP ───────────────────────────────────────────────────────────────────
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST  = 0.1
RANDOM_STATE   = 42

# ── Dynamic clustering ─────────────────────────────────────────────────────
# The notebook scans every k in K_RANGE and picks the one with the highest
# silhouette score automatically.
# Set FORCE_K to an int to skip the search and use that k directly.
K_RANGE = (4, 20)   # (min_k, max_k) inclusive
FORCE_K = None      # e.g. FORCE_K = 8 to lock a specific k

# ── Cluster -> node mapping ────────────────────────────────────────────────
# Fill this AFTER inspecting clusters in Cell 8.
# Multiple cluster ids can share the same node name (they are merged).
# Leave as empty dict to auto-name nodes 'cluster_0', 'cluster_1', ...
CLUSTER_TO_NODE: dict[int, str] = {
    # 0: "climate",
    # 1: "technology",
    # 2: "life_sciences",
    # 3: "society",
}

print("Config OK")


Config OK


## 1 — Load corpus


In [ ]:
import json
import pandas as pd

records = []
for path in sorted(CORPUS_DIR.glob("*.json")):
    try:
        doc = json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        records.append({"file": path.name, "url": "", "title": "",
                        "body": "", "topic_prefix": "PARSE_ERROR",
                        "parse_error": str(e)})
        continue
    name         = path.name
    topic_prefix = name.split("__")[0] if "__" in name else "(no prefix)"
    records.append({
        "file":         name,
        "url":          doc.get("url", ""),
        "title":        doc.get("title", ""),
        "body":         doc.get("body", ""),
        "topic_prefix": topic_prefix,
        "parse_error":  "",
    })

df = pd.DataFrame(records)
print(f"Total files: {len(df)}")
print()
print("Files per topic prefix:")
print(df["topic_prefix"].value_counts().to_string())


## 2 — Clean & normalise text

Fixes encoding artefacts (mojibake), strips residual HTML, and removes JS boilerplate
left by the scraper. Inspect the before/after examples to verify correctness.


In [ ]:
import re
import html

# Mojibake: UTF-8 characters that were decoded as Latin-1 during scraping.
# We map the garbled byte sequences back to the correct Unicode characters.
_MOJIBAKE = {
    "\u00c2\u00b6": "\u00b6",          # paragraph sign
    "\u00e2\u20ac\u2122": "'",         # right single quotation
    "\u00e2\u20ac\u0153": '"',         # left double quotation
    "\u00e2\u20ac\u009d": '"',         # right double quotation
    "\u00e2\u20ac\u201c": "\u2013",    # en dash
    "\u00e2\u20ac\u201d": "\u2014",    # em dash
    "\u00e2\u20ac\u00a6": "\u2026",    # ellipsis
    "\u00c2\u00a9": "\u00a9",          # copyright
    "\u00c2\u00ae": "\u00ae",          # registered trademark
    "\u00c2\u00b7": "\u00b7",          # middle dot
    "\u00c2\u00a0": " ",               # non-breaking space
}

# Boilerplate phrases that indicate a failed or partial scrape.
_BOILERPLATE = re.compile(
    r"Loading component\.{0,3}"
    r"|Past Events\s+View Upcoming"
    r"|This content is now at"
    r"|Please enable JavaScript"
    r"|To view this (video|page) please enable"
    r"|Cookie\s+(Policy|Settings|Notice)"
    , re.IGNORECASE,
)

def clean_text(text: str) -> str:
    if not text:
        return ""
    text = html.unescape(text)                       # &amp; &lt; etc.
    for bad, good in _MOJIBAKE.items():              # fix mojibake
        text = text.replace(bad, good)
    text = re.sub(r"<[^>]{1,200}>", " ", text)      # strip HTML tags
    text = _BOILERPLATE.sub(" ", text)              # remove boilerplate
    text = re.sub(r"[ \t]{2,}", " ", text)          # collapse spaces
    text = re.sub(r"\n{3,}", "\n\n", text)          # collapse blank lines
    return text.strip()


df["title_clean"] = df["title"].apply(clean_text)
df["body_clean"]  = df["body"].apply(clean_text)
df["body_len"]    = df["body_clean"].str.len()

# Spot-check: show a few files where cleaning changed something.
SPOT_CHECK_N = 3
changed = df[df["body_clean"] != df["body"]].head(SPOT_CHECK_N)
for _, row in changed.iterrows():
    print(f"FILE: {row['file']}")
    print(f"  BEFORE: {row['body'][:200]!r}")
    print(f"  AFTER : {row['body_clean'][:200]!r}")
    print()
print(f"Total docs with cleaning changes: {len(df[df['body_clean'] != df['body']])}")


## 3 — Filter bad documents


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(df["body_len"].clip(upper=5000), bins=60, edgecolor="white")
ax.axvline(MIN_BODY_LEN, color="red", linestyle="--",
           label=f"MIN_BODY_LEN={MIN_BODY_LEN}")
ax.set_xlabel("Body length after cleaning (chars, capped at 5 000)")
ax.set_ylabel("Count")
ax.set_title("Body-length distribution")
ax.legend()
plt.tight_layout(); plt.show()

bad = df[df["body_len"] < MIN_BODY_LEN]
print(f"\nDocuments below MIN_BODY_LEN ({MIN_BODY_LEN} chars): {len(bad)}")
bad[["file", "body_len", "body_clean"]]


In [ ]:
# Add filenames here to force-keep a document despite being below MIN_BODY_LEN.
FORCE_KEEP: set[str] = set()

clean = df[
    (df["body_len"] >= MIN_BODY_LEN) | df["file"].isin(FORCE_KEEP)
].copy()
clean = clean[clean["parse_error"] == ""].reset_index(drop=True)
excluded = df[~df["file"].isin(clean["file"])]

print(f"Excluded (bad scrapes) : {len(excluded)}")
print(f"Clean documents        : {len(clean)}")


## 4 — Embed documents (full text, no truncation)

Each document is encoded as `title + " " + full_body`.  
Embeddings are cached — delete `.embeddings_cache.npy` to force a fresh run.


In [ ]:
import numpy as np

texts = (clean["title_clean"] + " " + clean["body_clean"]).tolist()

reload = True
if EMBED_CACHE.exists():
    embeddings = np.load(EMBED_CACHE)
    if embeddings.shape[0] == len(texts):
        print(f"Loaded cached embeddings: {embeddings.shape}")
        reload = False
    else:
        print(f"Cache has {embeddings.shape[0]} rows but corpus has {len(texts)} — re-encoding.")

if reload:
    from sentence_transformers import SentenceTransformer
    print(f"Encoding {len(texts)} documents with '{EMBED_MODEL}' (full text) ...")
    model = SentenceTransformer(EMBED_MODEL)
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=16)
    np.save(EMBED_CACHE, embeddings)
    print(f"Saved -> {EMBED_CACHE}  shape={embeddings.shape}")


## 5 — UMAP: visualise by original topic prefix

Shows how well (or poorly) the original filename prefixes separate in embedding space.  
Overlapping colours = documents that likely don't belong to that prefix.


In [ ]:
import umap as umap_lib

reducer = umap_lib.UMAP(
    n_neighbors  = UMAP_NEIGHBORS,
    min_dist     = UMAP_MIN_DIST,
    random_state = RANDOM_STATE,
    verbose      = False,
)
xy = reducer.fit_transform(embeddings)
clean["umap_x"] = xy[:, 0]
clean["umap_y"] = xy[:, 1]

prefixes  = sorted(clean["topic_prefix"].unique())
cmap      = plt.cm.get_cmap("tab20", len(prefixes))
color_map = {p: cmap(i) for i, p in enumerate(prefixes)}

fig, ax = plt.subplots(figsize=(13, 8))
for prefix, grp in clean.groupby("topic_prefix"):
    ax.scatter(grp["umap_x"], grp["umap_y"],
               c=[color_map[prefix]], label=prefix, s=20, alpha=0.75)
ax.legend(markerscale=2, fontsize=8, loc="best", ncol=2)
ax.set_title("UMAP — coloured by original topic prefix")
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## 6 — Auto-select number of clusters

Scans every k in `K_RANGE` and picks the highest silhouette score automatically.  
Override with `FORCE_K` in the config cell once you know a good value.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

if FORCE_K is not None:
    best_k = FORCE_K
    print(f"FORCE_K={FORCE_K} — skipping auto-selection.")
else:
    k_min, k_max = K_RANGE
    k_range      = range(k_min, min(k_max + 1, len(clean) // 5))
    inertias, silhouettes = [], []

    print(f"Scanning k={k_min}..{k_max}", end="", flush=True)
    for k in k_range:
        km     = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init="auto")
        labels = km.fit_predict(embeddings)
        inertias.append(km.inertia_)
        sil    = silhouette_score(embeddings, labels,
                                  sample_size=min(600, len(clean)),
                                  random_state=RANDOM_STATE)
        silhouettes.append(sil)
        print(".", end="", flush=True)
    print(" done")

    best_idx = silhouettes.index(max(silhouettes))
    best_k   = list(k_range)[best_idx]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(list(k_range), inertias, marker="o")
    ax1.axvline(best_k, color="red", linestyle="--", label=f"auto k={best_k}")
    ax1.set_xlabel("k"); ax1.set_ylabel("Inertia")
    ax1.set_title("Elbow curve"); ax1.legend()

    ax2.plot(list(k_range), silhouettes, marker="o", color="green")
    ax2.axvline(best_k, color="red", linestyle="--", label=f"auto k={best_k}")
    ax2.set_xlabel("k"); ax2.set_ylabel("Silhouette (higher = better)")
    ax2.set_title("Silhouette scores"); ax2.legend()
    plt.tight_layout(); plt.show()

    print(f"\nAuto-selected k={best_k}  (silhouette={max(silhouettes):.3f})")
    print(f"To lock this: set FORCE_K={best_k} in the config cell.")


## 7 — Apply KMeans with auto-selected k


In [ ]:
km = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init="auto")
clean["cluster"] = km.fit_predict(embeddings)

print(f"Cluster sizes (k={best_k}):")
print(clean["cluster"].value_counts().sort_index().to_string())

cmap_c = plt.cm.get_cmap("tab20", best_k)
fig, ax = plt.subplots(figsize=(13, 8))
for cid, grp in clean.groupby("cluster"):
    node_name = CLUSTER_TO_NODE.get(cid, f"cluster_{cid}")
    ax.scatter(grp["umap_x"], grp["umap_y"],
               c=[cmap_c(cid)], label=f"{cid}: {node_name}", s=20, alpha=0.8)
ax.legend(markerscale=2, fontsize=8, loc="best")
ax.set_title(f"UMAP — KMeans clusters (k={best_k})")
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## 8 — Inspect clusters

Read the output below, then go back to the **config cell** and fill `CLUSTER_TO_NODE`,  
then re-run **Cell 7 onward** to see updated labels before exporting.


In [ ]:
TOP_N = 8  # sample docs printed per cluster

for cid in sorted(clean["cluster"].unique()):
    grp        = clean[clean["cluster"] == cid]
    node_label = CLUSTER_TO_NODE.get(cid, "(unnamed)")
    dominant   = grp["topic_prefix"].value_counts().head(3).to_dict()
    print(f"{chr(9472)*70}")
    print(f"Cluster {cid:2d}  ->  node: '{node_label}'   ({len(grp)} docs)")
    print(f"  Dominant prefix(es): {dominant}")
    for _, row in grp.head(TOP_N).iterrows():
        title = (row["title_clean"] or row["url"])[:90]
        print(f"  [{row['topic_prefix']}] {title}")
    print()


## 9 — Cross-tab: cluster vs original prefix


In [ ]:
ct = pd.crosstab(clean["cluster"], clean["topic_prefix"])
ct.index = [f"{c} ({CLUSTER_TO_NODE.get(c, '?')})" for c in ct.index]
print(ct.to_string())


## 10 — Export

Writes:
- **`cluster_assignments.json`** — `{filename: node_name}` for every clean document
- **`corpus_nodes_clustered.toml`** — drop-in replacement for `corpus_nodes.toml`

Then run: `python database_setup.py --config corpus_nodes_clustered.toml`


In [ ]:
import json as _json
from collections import defaultdict

node_files: dict[str, list[str]] = defaultdict(list)
assignments: dict[str, str]      = {}

for _, row in clean.iterrows():
    cid       = int(row["cluster"])
    node_name = CLUSTER_TO_NODE.get(cid, f"cluster_{cid}")
    node_files[node_name].append(row["file"])
    assignments[row["file"]] = node_name

OUTPUT_MANIFEST.write_text(
    _json.dumps(assignments, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {len(assignments)} entries -> {OUTPUT_MANIFEST}")

lines = [
    "# corpus_nodes_clustered.toml",
    f"# Auto-generated by data_preperation.ipynb  (k={best_k})",
    "# Do not edit by hand — re-run the notebook to regenerate.",
    "",
    "[corpus]",
    'dir = "corpus"',
    "",
    "[databases]",
    'dir = "dbs"',
    "",
]

all_files = sorted(clean["file"].tolist())
lines += ["[[node]]", 'name = "everything"']
lines.append("include = [")
for f in all_files:
    lines.append(f'  "{f}",')
lines += ["]", ""]

for node_name in sorted(node_files):
    files = sorted(node_files[node_name])
    lines += ["[[node]]", f'name = "{node_name}"']
    lines.append("include = [")
    for f in files:
        lines.append(f'  "{f}",')
    lines += ["]", ""]

OUTPUT_TOML.write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote {len(node_files)} nodes -> {OUTPUT_TOML}")
print()
print("Node summary:")
for node, files in sorted(node_files.items()):
    print(f"  {node:25s}  {len(files):4d} docs")
print(f"  {'EXCLUDED (bad scrapes)':25s}  {len(excluded):4d} docs")


## 11 — (Optional) Rebuild databases


In [ ]:
# Uncomment to rebuild all node SQLite FTS5 databases immediately.
# import subprocess, sys
# result = subprocess.run(
#     [sys.executable, "database_setup.py", "--config", str(OUTPUT_TOML)],
#     capture_output=True, text=True,
# )
# print(result.stdout)
# if result.returncode != 0:
#     print("STDERR:", result.stderr)
